# Pass 6A — BrEaST Dataset-Level Audit

Publication-grade audit of the independent **BrEaST-Lesions-USG** dataset before external model inference.

This pass validates metadata, locks the binary benign/malignant cohort, verifies physical image/tumor-mask integrity, and reproduces the BUSI raw-ROI construction for audit purposes only.

**No model inference, threshold tuning, retraining, or fine-tuning is performed.**

In [ ]:
from pathlib import Path
import hashlib, json, re
import cv2
import numpy as np
import pandas as pd
from PIL import Image

# Supplied workbook
CLINICAL_XLSX = Path("/mnt/data/f4a23bba-eca0-49e8-aa8d-70759cd9e08b.xlsx")

# EDIT THIS to the actual directory containing the downloaded BrEaST image/mask files.
DATASET_ROOT = Path("/path/to/BrEaST-Lesions-USG")

OUT_DIR = Path("results/pass6a_breast_dataset_audit")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
np.random.seed(SEED)

print("Workbook:", CLINICAL_XLSX)
print("Dataset root:", DATASET_ROOT)


In [ ]:
# Load documentation
assert CLINICAL_XLSX.exists(), f"Workbook not found: {CLINICAL_XLSX}"

xls = pd.ExcelFile(CLINICAL_XLSX)
print("Sheets:", xls.sheet_names)

clinical = pd.read_excel(
    CLINICAL_XLSX,
    sheet_name="BrEaST-Lesions-USG clinical dat"
)
dictionary = pd.read_excel(CLINICAL_XLSX, sheet_name="Data Dictionary")

print("Clinical shape:", clinical.shape)
print(clinical.columns.tolist())

required = [
    "CaseID", "Image_filename", "Mask_tumor_filename",
    "Classification", "Diagnosis", "Verification"
]
missing = [c for c in required if c not in clinical.columns]
assert not missing, f"Missing required columns: {missing}"

assert len(clinical) == 256
assert clinical["CaseID"].nunique() == 256
assert clinical["Image_filename"].nunique() == 256


In [ ]:
# Lock the binary external-validation cohort
# Normal cases are excluded because the current classifier is binary
# benign vs malignant and the four normal cases have no tumor masks.

binary = clinical[
    clinical["Classification"].isin(["benign", "malignant"])
].copy().reset_index(drop=True)

normal = clinical[clinical["Classification"].eq("normal")].copy()

print("Total documented cases:", len(clinical))
print("Binary cohort:", len(binary))
print(binary["Classification"].value_counts())
print("Excluded normal:", len(normal))

assert len(binary) == 252
assert binary["Classification"].value_counts().to_dict() == {
    "benign": 154,
    "malignant": 98
}
assert binary["Mask_tumor_filename"].notna().all()

binary["external_case_index"] = np.arange(len(binary))
binary.to_csv(OUT_DIR / "breast_binary_external_cohort.csv", index=False)

manifest = {
    "dataset": "BrEaST-Lesions-USG",
    "n_total_documented_cases": 256,
    "n_external_binary_cases": 252,
    "n_benign": 154,
    "n_malignant": 98,
    "excluded_normal": 4,
    "reason_for_exclusion": "Current classifier is binary benign/malignant; normal cases lack tumor masks.",
    "seed": SEED
}
with open(OUT_DIR / "breast_external_cohort_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)


In [ ]:
# Parse documented filenames.
def split_filename(value):
    if pd.isna(value):
        return []
    s = str(value).strip()
    if any(d in s for d in [";", "|", "\n", ","]):
        return [x.strip() for x in re.split(r"[;|,\n]+", s) if x.strip()]
    return [s] if s else []

binary["image_files_parsed"] = binary["Image_filename"].apply(split_filename)
binary["tumor_mask_files_parsed"] = binary["Mask_tumor_filename"].apply(split_filename)

print(binary["image_files_parsed"].map(len).value_counts())
print(binary["tumor_mask_files_parsed"].map(len).value_counts())

assert binary["image_files_parsed"].map(len).eq(1).all()
assert binary["tumor_mask_files_parsed"].map(len).eq(1).all()


In [ ]:
# Resolve physical files by basename.
# Ambiguous matches are rejected rather than silently choosing one.

assert DATASET_ROOT.exists(), (
    "Set DATASET_ROOT to the actual BrEaST dataset directory before running "
    "the physical-file audit."
)

all_files = [p for p in DATASET_ROOT.rglob("*") if p.is_file()]
basename_map = {}
for p in all_files:
    basename_map.setdefault(p.name, []).append(p)

def resolve_unique(name):
    matches = basename_map.get(str(name), [])
    if len(matches) == 0:
        return None
    if len(matches) > 1:
        raise RuntimeError(
            f"Ambiguous basename {name!r}: {len(matches)} physical matches."
        )
    return matches[0]

binary["image_path"] = binary["image_files_parsed"].str[0].map(resolve_unique)
binary["tumor_mask_path"] = binary["tumor_mask_files_parsed"].str[0].map(resolve_unique)

print("Missing images:", binary["image_path"].isna().sum())
print("Missing tumor masks:", binary["tumor_mask_path"].isna().sum())

assert binary["image_path"].notna().all()
assert binary["tumor_mask_path"].notna().all()


In [ ]:
# Physical image/mask integrity audit
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

records = []

for _, row in binary.iterrows():
    image_path = Path(row["image_path"])
    mask_path = Path(row["tumor_mask_path"])

    image = np.asarray(Image.open(image_path).convert("RGB"))
    mask = np.asarray(Image.open(mask_path).convert("L"))

    unique_mask = np.unique(mask)
    nonzero = mask > 0

    records.append({
        "CaseID": row["CaseID"],
        "Classification": row["Classification"],
        "image_exists": image_path.exists(),
        "mask_exists": mask_path.exists(),
        "image_width": image.shape[1],
        "image_height": image.shape[0],
        "mask_width": mask.shape[1],
        "mask_height": mask.shape[0],
        "dimensions_match": image.shape[:2] == mask.shape[:2],
        "mask_unique_value_count": len(unique_mask),
        "mask_unique_values_sample": ",".join(map(str, unique_mask[:10])),
        "mask_binary_like": len(unique_mask) <= 2,
        "mask_nonempty": bool(nonzero.any()),
        "mask_area_pixels": int(nonzero.sum()),
        "mask_area_fraction": float(nonzero.mean()),
        "image_sha256": sha256_file(image_path),
        "mask_sha256": sha256_file(mask_path)
    })

physical_audit = pd.DataFrame(records)

print("Dimension mismatches:", (~physical_audit["dimensions_match"]).sum())
print("Non-binary-like masks:", (~physical_audit["mask_binary_like"]).sum())
print("Empty masks:", (~physical_audit["mask_nonempty"]).sum())

assert physical_audit["dimensions_match"].all()
assert physical_audit["mask_binary_like"].all()
assert physical_audit["mask_nonempty"].all()


In [ ]:
# Reconstruct the exact raw-ROI operation used for BUSI.
# This is an audit-only reconstruction; no classifier is called.

kernel = cv2.getStructuringElement(
    cv2.MORPH_ELLIPSE, (15, 15)
)

def reconstruct_raw_roi(image_path, mask_path):
    image = np.asarray(Image.open(image_path).convert("RGB"))
    mask = np.asarray(Image.open(mask_path).convert("L"))

    binary_mask = (mask > 0).astype(np.uint8)
    dilated = cv2.dilate(binary_mask, kernel, iterations=1)

    ys, xs = np.where(dilated > 0)
    if len(xs) == 0:
        raise ValueError("Dilated tumor mask is empty.")

    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())

    masked = image.copy()
    masked[dilated == 0] = 0
    crop = masked[y0:y1+1, x0:x1+1]

    resized = cv2.resize(
        crop, (224, 224), interpolation=cv2.INTER_AREA
    )

    return x0, y0, x1, y1, crop, resized

roi_records = []

for _, row in binary.iterrows():
    x0, y0, x1, y1, crop, resized = reconstruct_raw_roi(
        row["image_path"], row["tumor_mask_path"]
    )

    roi_records.append({
        "CaseID": row["CaseID"],
        "Classification": row["Classification"],
        "bbox_x0": x0,
        "bbox_y0": y0,
        "bbox_x1": x1,
        "bbox_y1": y1,
        "bbox_width": x1 - x0 + 1,
        "bbox_height": y1 - y0 + 1,
        "crop_height": crop.shape[0],
        "crop_width": crop.shape[1],
        "resized_height": resized.shape[0],
        "resized_width": resized.shape[1]
    })

roi_audit = pd.DataFrame(roi_records)
assert (roi_audit["resized_height"] == 224).all()
assert (roi_audit["resized_width"] == 224)


In [ ]:
# Save final audit table and manifest
final_audit = (
    binary[
        [
            "CaseID", "Classification", "Image_filename",
            "Mask_tumor_filename", "Verification", "BIRADS",
            "image_path", "tumor_mask_path"
        ]
    ]
    .merge(
        physical_audit,
        on=["CaseID", "Classification"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        roi_audit,
        on=["CaseID", "Classification"],
        how="left",
        validate="one_to_one"
    )
)

audit_path = OUT_DIR / "breast_dataset_level_audit.csv"
final_audit.to_csv(audit_path, index=False)

summary = {
    "n_cases": len(final_audit),
    "n_benign": int((final_audit["Classification"] == "benign").sum()),
    "n_malignant": int((final_audit["Classification"] == "malignant").sum()),
    "all_images_present": bool(final_audit["image_exists"].all()),
    "all_masks_present": bool(final_audit["mask_exists"].all()),
    "all_dimensions_match": bool(final_audit["dimensions_match"].all()),
    "all_masks_binary_like": bool(final_audit["mask_binary_like"].all()),
    "all_masks_nonempty": bool(final_audit["mask_nonempty"].all()),
    "all_roi_reconstructed": bool(
        final_audit[
            ["bbox_width", "bbox_height",
             "resized_height", "resized_width"]
        ].notna().all().all()
    ),
    "mean_mask_area_fraction": float(final_audit["mask_area_fraction"].mean()),
    "median_mask_area_fraction": float(final_audit["mask_area_fraction"].median())
}

with open(OUT_DIR / "breast_dataset_level_audit_manifest.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Saved:", audit_path)


In [ ]:
# Publication-readiness gate
checks = {
    "252_binary_cases": len(final_audit) == 252,
    "154_benign": (final_audit["Classification"] == "benign").sum() == 154,
    "98_malignant": (final_audit["Classification"] == "malignant").sum() == 98,
    "unique_case_ids": final_audit["CaseID"].nunique() == 252,
    "images_present": final_audit["image_exists"].all(),
    "masks_present": final_audit["mask_exists"].all(),
    "dimensions_match": final_audit["dimensions_match"].all(),
    "binary_masks": final_audit["mask_binary_like"].all(),
    "nonempty_masks": final_audit["mask_nonempty"].all(),
    "roi_reconstruction": final_audit[
        ["bbox_width", "bbox_height",
         "resized_height", "resized_width"]
    ].notna().all().all()
}

for name, passed in checks.items():
    print(("PASS" if passed else "FAIL"), name)

assert all(checks.values()), "One or more dataset-audit gates failed."
print("\nPASS 6A COMPLETE")
